In [ ]:
from datasets import load_dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
from PIL import Image
import os

model_name = "microsoft/trocr-base-printed"
processor = TrOCRProcessor.from_pretrained(model_name)
model = VisionEncoderDecoderModel.from_pretrained(model_name)

# Custom Coeur d'Alene charset — make sure your tokenizer covers these
special_tokens = ["č", "ˡɫ", "ʷ","ᵃ\u0308", "u̥","ᵘ", "ɔ", "ä", "ĺ", "ý", "ɛ", "x̥","ʙ", "ẃ", "q́","ḿ", "ˠ", "‿", "t́", "ʀ"  ]  # add as needed
processor.tokenizer.add_tokens(special_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

In [ ]:

# Load dataset
def load_samples(path):
    data = {"image": [], "text": []}
    for fname in os.listdir(path):
        if fname.endswith(".png"):
            with open(os.path.join(path, fname.replace(".png", ".txt"))) as f:
                data["image"].append(Image.open(os.path.join(path, fname)).convert("RGB"))
                data["text"].append(f.read().strip())
    return data

train_data = load_samples("../cda/train")
val_data = load_samples("../cda/val")

from datasets import Dataset
train_ds = Dataset.from_dict(train_data)
val_ds = Dataset.from_dict(val_data)

In [ ]:
def preprocess(batch):
    pixel_values = processor(images=batch["image"], return_tensors="pt").pixel_values
    labels = processor.tokenizer(batch["text"], padding="max_length", truncation=True, return_tensors="pt").input_ids
    return {"pixel_values": pixel_values.squeeze(), "labels": labels.squeeze()}

train_ds = train_ds.map(preprocess, batched=False)
val_ds = val_ds.map(preprocess, batched=False)

In [ ]:

training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr-cda",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=10,
    logging_steps=100,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    predict_with_generate=True,
)




In [ ]:

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)


In [ ]:
trainer.train()

In [ ]:
model.eval()
for sample in val_data["image"]:
    pixel_values = processor(sample, return_tensors="pt").pixel_values
    generated_ids = model.generate(pixel_values)
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(text)


In [ ]:
model.save_pretrained("./trocr-cda")
processor.save_pretrained("./trocr-cda")


In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

processor = TrOCRProcessor.from_pretrained("./trocr-cda")
model = VisionEncoderDecoderModel.from_pretrained("./trocr-cda")

image = Image.open("../test/line_screenshot.png").convert("RGB")
pixel_values = processor(image, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)
text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(text)
